# Homework: your session 5 answers in pandas, and a decision diary, worked solution

Every answer in task 1 is pinned with an `assert` to the number in the
session 5 worked answers, so the two languages are checked against each
other.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

sys.path.insert(0, "sessions/session-06/solutions")
from cleaning import clean_plays, load_messy

plays = pd.read_csv("data/clean/plays.csv")
artists = pd.read_csv("data/clean/artists.csv")
awards = pd.read_csv("data/clean/awards.csv")

print(plays.shape, artists.shape, awards.shape)

# Task 1: the ten questions

## 1. How many artists?

In [ ]:
assert len(artists) == 40
print(len(artists), "artists")

`len()` on a DataFrame counts the rows, exactly as it did on the list of
dictionaries. `artists.shape[0]` is the same number.

## 2. Artists from the Netherlands

In [ ]:
dutch = artists[artists["country"] == "Netherlands"][["artist_name", "genre"]]
dutch = dutch.sort_values("artist_name")

assert list(dutch["artist_name"]) == ["DJ Kompas", "Neon Harbour", "The Long Commute"]
dutch

Filter the rows, then pick the columns, then sort. Three steps, in the order
they happen.

## 3. Artists with an unknown `formed_year`

In [ ]:
artists.dtypes

`formed_year` is `float64`, a decimal, even though every year is a whole
number. A missing value in pandas is `NaN`, and `NaN` is a special decimal
value, so any number column with a gap in it is stored as decimals. When you
see `2014.0` where you expected `2014`, look for missing values.

Now the trap. The SQL lesson was that `= NULL` finds nothing. The pandas
equivalent fails in the same silent way:

In [ ]:
print(len(artists[artists["formed_year"] == None]), "rows found with == None")   # noqa: E711

Zero, and no error. A missing value is not equal to anything, not even
`None`. The only way to ask is `isna()`, which is `IS NULL`:

In [ ]:
unknown = artists[artists["formed_year"].isna()][["artist_name", "country", "genre"]]

assert set(unknown["artist_name"]) == {"Rosewood Lane", "Cassette Revival"}
unknown

That is the first trap pandas shares with SQL. (The `# noqa` comment above only
tells code checkers that the `== None` is deliberate.)

## 4. The five longest plays

In [ ]:
longest = plays.sort_values("minutes_played", ascending=False).head(5)
longest = longest[["track_name", "played_at", "minutes_played"]]

assert list(longest["track_name"]) == [
    "Paper Avenue", "Harbour Season", "Broken Lines", "Static Wires", "Northern Window",
]
longest

`sort_values(...).head(5)` is `ORDER BY ... DESC LIMIT 5`. There is also a
shortcut, `plays.nlargest(5, "minutes_played")`, which gives the same rows.

## 5. Plays in 2024 and in 2025

In [ ]:
per_year = plays["played_at"].str[:4].value_counts().sort_index()

assert per_year["2024"] == 1122
assert per_year["2025"] == 1061
per_year

`.str[:4]` slices every value in the column, the same as
`substr(played_at, 1, 4)` and the same as `date[:4]` on one string in
session 4. `.sort_index()` puts the years in order; without it,
`value_counts` would put the bigger count first.

With real dates, as in notebook 05, it reads as what it means:

In [ ]:
by_year = pd.to_datetime(plays["played_at"], format="%Y-%m-%d").dt.year.value_counts().sort_index()
assert by_year.to_dict() == {2024: 1122, 2025: 1061}
by_year


## 6. Tracks played more than 40 times

In [ ]:
track_counts = plays["track_name"].value_counts()
popular = track_counts[track_counts > 40]

assert list(popular.index) == ["Broken Lines", "Salt Lines", "Amber Ferry", "Broken Mornings", "Open Orbit"]
assert list(popular) == [67, 63, 55, 52, 44]
popular

Count, then filter the counts: that is `HAVING COUNT(*) > 40`, with no
special keyword.

## 7. Total minutes per device, to the nearest minute

In [ ]:
device_minutes = plays.groupby("device")["minutes_played"].sum().round().astype(int)
device_minutes = device_minutes.sort_values(ascending=False)

assert device_minutes.to_dict() == {"phone": 4308, "laptop": 2144, "speaker": 1335, "car": 778, "tablet": 416}
device_minutes

`.round()` with nothing in the brackets rounds to whole numbers but leaves
them as decimals: `4308.0`. `.astype(int)` changes the type, so they print
as `4308`. SQL's `ROUND` does the same thing, and DB Browser hides it.

## 8. Genres with five or more artists

In [ ]:
genre_artists = artists["genre"].value_counts()
big_genres = genre_artists[genre_artists >= 5]

assert big_genres.to_dict() == {"Electronic": 7, "Indie": 6, "Folk": 5, "Jazz": 5, "Latin": 5, "Rock": 5}
big_genres

Notice the order of the four genres with 5. `value_counts` sorts by the
count, and among a tie the order is whatever it happens to be. In SQL you
added `genre` to the `ORDER BY` to settle it. The pandas equivalent sorts by
two things at once:

In [ ]:
tidy = big_genres.rename("artists").reset_index()
tidy.sort_values(["artists", "genre"], ascending=[False, True])

## 9. Skip rate per device

This is the trap SQL had and pandas does not.

In [ ]:
skips = plays.groupby("device").agg(
    plays=("play_id", "count"),
    skipped=("skipped", "sum"),
)
skips["skipped_pct"] = (100 * skips["skipped"] / skips["plays"]).round(1)
skips = skips.sort_values("skipped_pct", ascending=False)

assert skips["skipped_pct"].to_dict() == {"speaker": 14.0, "tablet": 13.3, "laptop": 11.5, "phone": 11.4, "car": 10.6}
skips

In SQLite, `19 / 179` was `0`, and you needed `100.0` to force a proper
division. In Python 3, `/` always gives the decimal answer, so the trap is
simply not there.

A new column is made by assigning to it: `skips["skipped_pct"] = ...`, the
way you added a key to a dictionary.

And a shortcut worth knowing: `skipped` holds only 0s and 1s, so its
**average** is the fraction of plays skipped. `mean` gives you the rate
directly:

In [ ]:
rate = (100 * plays.groupby("device")["skipped"].mean()).round(1)
assert rate.to_dict() == skips["skipped_pct"].to_dict()
rate.sort_values(ascending=False)

## 10. The busiest single day

In [ ]:
per_day = plays["played_at"].value_counts()

assert per_day.index[0] == "2024-03-24"
assert per_day.iloc[0] == 11
per_day.head(5)

2024-03-24 with 11 plays. And, as in session 5, look at second place before
believing it: 10, then three days on 8. A "busiest day" that wins by one play
out of 2,183 is not much of a finding.

`head(5)` instead of `head(1)` costs nothing, and shows you that. This is
the second trap pandas shares with SQL: no notation protects you from a
winner that is not really a winner.

## Your join-type question, an example

The example from the session 5 worked answers: on average, how many awards
does an artist in each genre have?

In [ ]:
def awards_per_artist(how):
    """Return awards per artist for each genre, with the given kind of merge."""
    merged = artists.merge(awards, on="artist_id", how=how)
    per_genre = merged.groupby("genre").agg(
        artists=("artist_id", "nunique"),
        awards=("award_id", "count"),
    )
    per_genre["per_artist"] = (per_genre["awards"] / per_genre["artists"]).round(2)
    return per_genre.sort_values("per_artist", ascending=False)


inner = awards_per_artist("inner")
inner

In [ ]:
left = awards_per_artist("left")

assert len(inner) == 7 and len(left) == 8
assert inner.loc["Rock", "per_artist"] == 3.0 and left.loc["Rock", "per_artist"] == 0.6
assert left.loc["Afrobeat", "awards"] == 0
left

Seven genres against eight, and Rock at 3.0 against 0.6: the same two
answers as the SQL. The left merge answers the question. The inner merge
quietly asks a different one, "how many awards does an award **winner**
have": Rock's one winner, with 3 awards, becomes the headline, and Afrobeat,
whose 3 artists have won nothing, vanishes. "0 awards" is an answer, not a
missing row.

`nunique` counts distinct values, `COUNT(DISTINCT a.artist_id)`. `count` on
`award_id` is `COUNT(w.award_id)`: it skips the `NaN` that the left merge
leaves for an artist with no awards, so they count 0 and not 1. The
`size` versus `count` lesson from notebook 02, again.

# Which was easier?

My answer, for comparison with yours:

* **Easier in pandas:** 5 (`.str[:4]` or `.dt.year` beats remembering
  `strftime`), 6 and 8 (no `HAVING`), 9 (no integer division to trip over).
* **Easier in SQL:** 2, slightly. `WHERE country = 'Netherlands'` reads
  more like English than the double brackets do.
* **About the same:** the rest, including the join-type question. Same
  ideas, different punctuation.

Neither language is better. They are two notations for one set of ideas,
and you now read both.

# Task 2: which device gets skipped most?

In [ ]:
raw = load_messy("data/messy/plays_messy.csv")
cleaned = clean_plays(raw)
print(len(raw), "rows messy,", len(cleaned), "rows cleaned")

In [ ]:
def skip_rates(df):
    """Return one row per device: number of plays, and percentage skipped, highest first."""
    rates = df.groupby("device")["skipped"].agg(plays="size", rate="mean")
    rates["rate"] = rates["rate"] * 100
    return rates.sort_values("rate", ascending=False)


def winner(rates):
    """Return a one-line summary of the top row of a skip_rates table."""
    device = rates.index[0]
    return f"{device!r}: {rates['rate'].iloc[0]:.1f}% of {rates['plays'].iloc[0]} plays"

`{device!r}` prints the value with quotes round it, the way `repr()`
does. When the values themselves are the problem, like `"CAR"` versus
`"car"` or `"phone "` with a space, quotes are the only way to see it.

### Step 1: as loaded

In [ ]:
step1 = skip_rates(raw)
print(winner(step1))
step1.head()

`"CAR"` in capitals, at 17.2%, from 29 plays. Every spelling of every
device is its own group, so the "winner" is just whichever small group
happened to have a few skips in it. Further down, `"tablet"` with 6 plays
is at 16.7%.

### Step 2: device text cleaned, nothing else

In [ ]:
step2_data = raw.copy()
step2_data["device"] = step2_data["device"].str.strip().str.lower()
step2_data = step2_data[step2_data["device"] != ""]

step2 = skip_rates(step2_data)
print(winner(step2))
step2

Five groups, and speaker wins at 14.1%. **This is the step that changed the
answer.**

### Step 3: duplicates dropped too

In [ ]:
step3 = skip_rates(step2_data.drop_duplicates())
print(winner(step3))
step3

Still speaker, still 14.1%. The 40 copies were spread across devices, so
removing them moved the counts (1,024 phone plays became 1,004) but hardly
moved the rates. Dropping duplicates changes totals far more than it
changes percentages.

### Step 4: the full `clean_plays`

In [ ]:
step4 = skip_rates(cleaned)
print(winner(step4))
step4

Now `"unknown"` wins, at 15.9% of 69 plays.

### Step 5: the clean file

In [ ]:
step5 = skip_rates(plays)
print(winner(step5))
step5

In [ ]:
assert step1.index[0] == "CAR"
assert step2.index[0] == step3.index[0] == step5.index[0] == "speaker"
assert step4.index[0] == "unknown"
assert round(step4.loc["speaker", "rate"], 1) == 14.1
assert round(step5.loc["speaker", "rate"], 1) == 14.0

### The diary

| Step | Most skipped | Rate | Plays |
|---|---|---|---|
| 1. as loaded | `CAR` | 17.2% | 29 |
| 2. text cleaned | speaker | 14.1% | 333 |
| 3. duplicates dropped | speaker | 14.1% | 326 |
| 4. full clean, with "unknown" | unknown | 15.9% | 69 |
| 5. the clean file | speaker | 14.0% | 336 |

**1. Which decision changed the answer most?** Cleaning the text. Before
it, the question was not really being asked: you were ranking twenty
spellings, not five devices. After it, every later step picked the same
winner as the clean file, within 0.1 of a percentage point, as long as
`"unknown"` stays out of the ranking.

**2. What to do about `"unknown"`.** Keep it in the table, and keep it out
of the ranking. It is not a device; it is 69 plays on some mixture of
devices. "The most skipped device is unknown" is a sentence with no
meaning. The honest report is: "speaker is skipped most, at 14.1%; 69
plays (3%) had no device recorded." Labelling the gap kept it visible,
which is the point of labelling it. It did not turn it into a category you
can compare.

**3. Would `min_plays` have saved you?** The session 4 stretch only ranked
groups with enough plays to mean something. With a minimum of 30 plays,
`"CAR"` (29) and `"tablet"` (6) drop out of step 1. The new winner is
`"LAPTOP"`, in capitals, at 16.2% of 74 plays. Also wrong: in the clean file,
laptop's rate is 11.5%.

A minimum protects you from small groups. It cannot protect you from
asking the question about spellings instead of devices. Only cleaning does
that.

In [ ]:
step1_big = step1[step1["plays"] >= 30]
print(winner(step1_big))
assert step1_big.index[0] == "LAPTOP"